# Test the YOLO + CNN cascade — side-by-side

**What this notebook answers:** is the second-stage CNN classifier earning its keep on this floor plan? How many bboxes is it rejecting, and do the rejections actually look like non-columns?

**How it works:** one inference run, two renders.

1. YOLO on tiles → post-process (aspect / size / shape / OCR-skip / NMS) → `boxes_pre_classifier`.
2. CNN classifier on the survivors → `boxes_kept` (P ≥ threshold) + `boxes_rejected` (P < threshold).
3. Output: `<plan>_yolo_only.png` (red boxes = pre-classifier) AND `<plan>_yolo_cnn.png` (green = kept, dashed grey = rejected). Side-by-side preview, zoom-crop comparison, and a 16-tile gallery of rejected boxes with their `P(column)` for threshold tuning.

**Missing `column_classifier.pt`?** The notebook still runs — cell 6 prints a note and skips the cascade-only renders. Train it via the 🧠 Train CNN button in column-review, or `python3 scripts/train_bbox_classifier.py`.

In [ ]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw
from ultralytics import YOLO
import matplotlib.pyplot as plt

Image.MAX_IMAGE_PIXELS = None  # A0 plans exceed Pillow's default cap


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────
DETECT_WEIGHTS       = Path('column_detect.pt')
CLASSIFIER_WEIGHTS   = Path('column_classifier.pt')
CLASSIFIER_THRESHOLD = 0.5     # tune in cell 6 without re-running YOLO

IMAGE_PATH = Path('/home/jiezhi/Documents/TGCH floor plan/L3.jpg')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Tile geometry — matches generate_column.py + train.py + column-review.
TILE_SIZE = 1280
TILE_STEP = 1080            # 200 px overlap
CONF_TH   = 0.25
IOU_TH    = 0.45
INPUT_DPI = 300             # rasterisation DPI of the source PNG/JPG
DEVICE    = 0 if torch.cuda.is_available() else 'cpu'

print(f'detector   : {DETECT_WEIGHTS}')
print(f'classifier : {CLASSIFIER_WEIGHTS} (threshold={CLASSIFIER_THRESHOLD})')
print(f'image      : {IMAGE_PATH}')
print(f'device     : {DEVICE}')


In [ ]:
# ── LOAD MODEL + IMAGE ───────────────────────────────────────────────────
# Walk up from cwd to find scripts/ so the notebook is portable.
import sys
from pathlib import Path as _Path
ROOT = next(
    (p for p in [_Path.cwd(), *_Path.cwd().parents]
     if (p / 'scripts' / 'tiled_inference.py').exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        f'Could not locate scripts/tiled_inference.py walking up from {_Path.cwd()}'
    )
for d in (ROOT / 'scripts', ROOT):
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))

# Resolve relative weight paths against ROOT so the notebook works from
# any cwd. Absolute paths in cell 2 are left alone.
if not DETECT_WEIGHTS.is_absolute():
    DETECT_WEIGHTS = ROOT / DETECT_WEIGHTS
if not CLASSIFIER_WEIGHTS.is_absolute():
    CLASSIFIER_WEIGHTS = ROOT / CLASSIFIER_WEIGHTS

detector = YOLO(str(DETECT_WEIGHTS))
img      = Image.open(IMAGE_PATH).convert('RGB')
W, H     = img.size
classifier_available = CLASSIFIER_WEIGHTS.is_file()
print(f'image size           : {W} × {H} px')
print(f'classifier available : {classifier_available} ({CLASSIFIER_WEIGHTS})')


In [ ]:
# ── TILED INFERENCE ──────────────────────────────────────────────────────
# scripts/tiled_inference.tiled_predict is the single source of truth for
# tile geometry + white-padding + per-tile count tracking. column-review
# (production) uses the same helper.
from tiled_inference import tiled_predict, n_tiles_for_image

all_boxes, all_scores, tile_counts = tiled_predict(
    detector, img,
    tile=TILE_SIZE, step=TILE_STEP,
    conf=CONF_TH, iou=IOU_TH,
    device=DEVICE,
)
n_grid = n_tiles_for_image(W, H, tile=TILE_SIZE, step=TILE_STEP)
print(f'tiles processed         : {len(tile_counts)}  (grid: {n_grid})')
print(f'raw detections (pre-NMS): {len(all_boxes)}')


In [ ]:
# ── POST-PROCESS (classifier OFF; cell 6 applies it separately) ─────────
from postprocess_pipeline import run_pipeline, DEFAULT_CONFIG, format_audit
from ood_detector import OutOfDistributionError

img_gray = np.asarray(img.convert('L'))
try:
    boxes_pre_classifier, scores_pre_classifier, audit = run_pipeline(
        img_gray, all_boxes, all_scores,
        config=DEFAULT_CONFIG,
        input_dpi=INPUT_DPI,
        tile_detection_counts=tile_counts,
    )
except OutOfDistributionError as e:
    print(f'OOD HARD FAILURE: {e}')
    raise

print(format_audit(audit))
print()
print(f'post-process survivors  : {len(boxes_pre_classifier)}')


In [ ]:
# ── CLASSIFIER SCORING + HISTOGRAM ───────────────────────────────────────
# Split the post-NMS survivors into kept vs rejected so cells 8-11 can
# show what the CNN actually removed. If the .pt is missing, fall back
# to keeping everything and skip the histogram.
if classifier_available and len(boxes_pre_classifier):
    from column_review.bbox_classifier import predict_batch
    probs, keep = predict_batch(
        img_gray, boxes_pre_classifier,
        weights_path=CLASSIFIER_WEIGHTS,
        threshold=CLASSIFIER_THRESHOLD,
    )
    boxes_kept      = boxes_pre_classifier [keep]
    scores_kept     = scores_pre_classifier[keep]
    boxes_rejected  = boxes_pre_classifier [~keep]
    scores_rejected = scores_pre_classifier[~keep]
    probs_rejected  = probs[~keep]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(probs, bins=40, color='#1976D2', edgecolor='white')
    ax.axvline(CLASSIFIER_THRESHOLD, color='#D32F2F', linestyle='--',
               label=f'threshold = {CLASSIFIER_THRESHOLD}')
    ax.set_xlabel('P(column)')
    ax.set_ylabel('count')
    ax.set_title('CNN classifier probability across post-NMS survivors')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    # No classifier file → pass-through. Cells 8-11 detect the empty
    # rejected set and skip the cascade-only renders cleanly.
    probs           = np.ones (len(boxes_pre_classifier), dtype=np.float32)
    boxes_kept      = boxes_pre_classifier
    scores_kept     = scores_pre_classifier
    boxes_rejected  = np.zeros((0, 4), dtype=np.float32)
    scores_rejected = np.zeros((0,),    dtype=np.float32)
    probs_rejected  = np.zeros((0,),    dtype=np.float32)
    reason = ('disabled — no post-process survivors'
              if not len(boxes_pre_classifier)
              else f'classifier weights missing at {CLASSIFIER_WEIGHTS}')
    print(f'CNN stage skipped ({reason}) — keeping all {len(boxes_kept)} boxes.')
    print('Train the classifier via the 🧠 Train CNN button or '
          '`python3 scripts/train_bbox_classifier.py`.')


In [ ]:
# ── STAGE-WISE COUNT TABLE ───────────────────────────────────────────────
n_raw  = len(all_boxes)
n_pre  = len(boxes_pre_classifier)
n_kept = len(boxes_kept)
n_rej  = len(boxes_rejected)

def _pct(num, denom):
    return f'({100 * num / denom:+.1f}%)' if denom else '(—)'

print(f"{'stage':<28}{'count':>8}   {'delta':>10}")
print('─' * 50)
print(f"{'YOLO raw':<28}{n_raw:>8}")
print(f"{'post-process kept':<28}{n_pre:>8}   {_pct(n_pre - n_raw, n_raw):>10}")
print(f"{'classifier kept':<28}{n_kept:>8}   {_pct(n_kept - n_pre, n_pre):>10}")
print(f"{'classifier rejected':<28}{n_rej:>8}   {_pct(-n_rej, n_pre):>10}")
print('─' * 50)
print(f"{'FINAL':<28}{n_kept:>8}")


In [ ]:
# ── TWO ANNOTATED PNGs ───────────────────────────────────────────────────
# yolo_only: what the pipeline would emit WITHOUT the classifier.
# yolo_cnn:  green = kept, dashed grey = rejected, so the viewer can see
#            exactly which boxes the CNN removed and where they were.
lw = max(2, min(W, H) // 1500)

def _draw_rects(im, boxes, *, color, width, dashed=False):
    draw = ImageDraw.Draw(im)
    if not dashed:
        for (x1, y1, x2, y2) in boxes:
            draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        return
    # Dashed: stride 12 / gap 6 along each edge.
    stride, gap = 12, 6
    for (x1, y1, x2, y2) in boxes:
        for x in range(int(x1), int(x2), stride + gap):
            draw.line([(x, y1), (min(x + stride, x2), y1)], fill=color, width=width)
            draw.line([(x, y2), (min(x + stride, x2), y2)], fill=color, width=width)
        for y in range(int(y1), int(y2), stride + gap):
            draw.line([(x1, y), (x1, min(y + stride, y2))], fill=color, width=width)
            draw.line([(x2, y), (x2, min(y + stride, y2))], fill=color, width=width)

# Render 1: YOLO + post-process only (red).
yolo_only = img.copy()
_draw_rects(yolo_only, boxes_pre_classifier, color=(220, 30, 30), width=lw)
path_yolo = OUTPUT_DIR / f'{IMAGE_PATH.stem}_yolo_only.png'
yolo_only.save(path_yolo, optimize=True)

# Render 2: cascade (green = kept, dashed grey = rejected).
yolo_cnn = img.copy()
if len(boxes_rejected):
    _draw_rects(yolo_cnn, boxes_rejected, color=(140, 140, 140), width=lw, dashed=True)
_draw_rects(yolo_cnn, boxes_kept, color=(34, 160, 60), width=lw)
path_cnn = OUTPUT_DIR / f'{IMAGE_PATH.stem}_yolo_cnn.png'
yolo_cnn.save(path_cnn, optimize=True)

print(f'saved → {path_yolo.resolve()}')
print(f'saved → {path_cnn.resolve()}')


In [ ]:
# ── SIDE-BY-SIDE PREVIEW ─────────────────────────────────────────────────
# Downscale to keep the figure light; the saved PNGs above are full-res.
thumb_yolo = yolo_only.copy(); thumb_yolo.thumbnail((2400, 2400))
thumb_cnn  = yolo_cnn.copy();  thumb_cnn.thumbnail((2400, 2400))

fig, axes = plt.subplots(1, 2, figsize=(20, 11))
axes[0].imshow(thumb_yolo)
axes[0].set_title(f'YOLO + post-process only — {n_pre} boxes (red)')
axes[0].axis('off')
axes[1].imshow(thumb_cnn)
axes[1].set_title(
    f'YOLO + CNN cascade — {n_kept} kept (green), {n_rej} rejected (dashed grey)'
)
axes[1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── ZOOM-CROP COMPARISON ─────────────────────────────────────────────────
# Pick 4 regions that each contain at least one rejected box so the
# viewer can spot-check whether the rejection makes visual sense.
# Falls back to random crops if there are no rejections.
import random
random.seed(0)
crop_size = 1500

if len(boxes_rejected):
    # Centre each crop on a randomly-chosen rejected box.
    sample = random.sample(
        range(len(boxes_rejected)),
        k=min(4, len(boxes_rejected)),
    )
    centres = []
    for i in sample:
        x1, y1, x2, y2 = boxes_rejected[i]
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        centres.append((cx, cy))
    # Pad with random centres if fewer than 4.
    while len(centres) < 4:
        centres.append((random.randint(0, W), random.randint(0, H)))
else:
    centres = [(random.randint(0, W), random.randint(0, H)) for _ in range(4)]

fig, axes = plt.subplots(4, 2, figsize=(14, 24))
for row, (cx, cy) in enumerate(centres):
    x0 = max(0, min(W - crop_size, cx - crop_size // 2))
    y0 = max(0, min(H - crop_size, cy - crop_size // 2))
    axes[row, 0].imshow(yolo_only.crop((x0, y0, x0 + crop_size, y0 + crop_size)))
    axes[row, 0].set_title(f'YOLO-only @ ({x0}, {y0})')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(yolo_cnn.crop((x0, y0, x0 + crop_size, y0 + crop_size)))
    axes[row, 1].set_title(f'YOLO+CNN @ ({x0}, {y0})')
    axes[row, 1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ── REJECTED-BOX GALLERY ─────────────────────────────────────────────────
# Up to 16 of the boxes the CNN dropped, each as a 64×64 crop next to
# its P(column). If rejected boxes still look like columns, the threshold
# is too high — drop CLASSIFIER_THRESHOLD in cell 2 and rerun cells 6-11.
from column_review.bbox_classifier import crop_64x64

if not len(boxes_rejected):
    print('no rejected boxes — the classifier is permissive at threshold '
          f'{CLASSIFIER_THRESHOLD}. Either every survivor IS a column, or '
          'the threshold is too low.')
else:
    # Sort by P(column) descending so the BORDERLINE rejections (just
    # below threshold) come first — those are the most informative for
    # threshold tuning.
    n_show = min(16, len(boxes_rejected))
    order  = np.argsort(-probs_rejected)[:n_show]
    rows = (n_show + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(14, 3.5 * rows))
    axes = np.atleast_2d(axes)
    for k, idx in enumerate(order):
        r, c = k // 4, k % 4
        crop = crop_64x64(img_gray, boxes_rejected[idx])
        axes[r, c].imshow(crop, cmap='gray', vmin=0, vmax=255)
        axes[r, c].set_title(f'P(column)={probs_rejected[idx]:.2f}')
        axes[r, c].axis('off')
    # Hide unused axes in the last row.
    for k in range(n_show, rows * 4):
        r, c = k // 4, k % 4
        axes[r, c].axis('off')
    plt.tight_layout(); plt.show()
    print(f'showing {n_show} of {len(boxes_rejected)} rejected boxes '
          f'(highest-P first).')
